IMPLEMENTAZIONE DI YOLOv8: DALLA TEORICA ALLA PRATICA CON ULTRALYTICS

YOLOv8 è una famiglia di modelli Ultralytics rilasciata nel 2023, progettata per task come object detection, segmentazione, pose, oriented bounding box e classificazione. Per la detection esistono varianti, n, s, m, l, x, con compromessi diversi tra velocità e accuratezza.

Il flusso è:
immagine/video -> YOLOv8 -> bounding box, classe, confidence -> output strutturato.
 
Con Ultralytic non devi implementare YOLO 'a mano'. 
Usi una libreria che gestisce:
caricamento modello, inference, training, validation, export, tracking
Ultralytic espone infatti modalità distinte come:
- train - val - predict - export - track - bencmnark 

Ci sono diverse varianti di yolov8:
- yolov8n: nano, più leggere e veloce
- yolov8s: small
- yolov8m: medium
- yolov8l: large
- yolov8x: extra large, più grande e generalmente più accurato

YOLOv8 detection pre-addestrato usa normalmente i pesi addestrati sul dataset COCO, che contiene 80 classi di oggetti.
E' la stessa distinzione fatta per MobileNet/ResNet e ImageNet
YOLO modello/famiglia
COCO dataset di pre-trainig

COCO

COCO è un grande dataset di raccolta immagini usate per addestrare e valutare modelli di Computer Vision.
Il nome significa Common Objects in Context. L'idea non è mostrare oggetti isolati su sfondo pulito, ma oggetti comuni dentro scene reali, per es. una persona in strada, una bici vicino a un'auto, ecc. Il paper originale nasce con l'obbiettivo di migliorare il riconoscimento dentro scene complesse e la comprensione del contesto.

COCO = immagini + etichette (usate per insegnare al modello cosa riconoscere)
YOLO = modello che impare dai dati COCO

COCO contiene circa 330.000 immagini, 200.000 annotate, circa 1,5 milioni di istanze di oggetti e 80 categorie principali usate per object detection. Le classi comprendono oggetti comuni come: persona, bicicletta, macchina, moto, cane, gatto, cellulare, ecc
Se dai a COCO una foto con persona, macchina e cane, il modello può già restituire qualcosa come_ persona 0.97, cane 0.91, macchina 0.88, senza che tu debba addestrare nulla.
;a se gli dai in paso una foto con oggetti mai visti, non classificati e non facenti parte delle 80 categorie che COCO ha individuato, allora COCO non riconosce quella classe. Non perchè YOLO sia incapace, ma perchè, durante il training, nessuno ha insegnato che quell'oggetto di chiama 'xyz'
Ed è qui che entra in gioco il tuo dataset custom (tramite transfer learning)

con l'istruzione: model=YOLO("yolov8n.pt")
puoi già riconoscere classi come: persone, macchine, cani, gatti, biciclette, bus, ...
senza alcun tipo di training aggiuntivo

Per fare una predizione
from ultralytics import YOLO
model=YOLO("yolov8n.pt")
result=model.predict(sourcce="mia_immagine.jpg", conf=0.5)
Ultralytics restituisce una lista di oggetti Results, contenenti le detecion prodotte dal modello. La modalità Predict accetta, directory, video, array NumPy, webcam, e altre sorgenti.

Dopo il predict puoi leggere il risultato con
print(result[0].boxes)
restituisce coordinate, classe, confidence

al .predict, invece di passare l'immagine, puoi passare la webcam
model.predict(source=0, show=True)
soource=0 indica la webcam locale.

In questo modo si arriva al real-time object-detection

Adesso la parte più interessante: addestrare YOLO sui tuoi oggetti
Immagina di voler riconoscere: raccordo, valvola, tubo
COCO non conosce neccessariamente le tue specifiche classi aziendali
Devi costruire un dataset personalizzato
Ogni immagine ha associati due file, nel primo indichi le coordinate normalizzaet (classe_id, x_center, y_centerm, width height)
Poi un file YAML che dice dove sono i dati + quali sono le classi
Fai transfer learning
model=YOLO("yolov8n.pt")
model.traing(data="dataset.yaml", epochs=50, imgsz=640)

Quindi non parti da zero, YOLOv8 è gia addestrato su COCO, conosce feature visive generiche.
tracendo il train sul tuo dataset impara cos'è un raccordo, cos'è una valvola e un tubo.

Dopo il training normalmente avrai checkpoint come, best.pt, e last.pt
Per l'inferenza userei il migliore "best.pt"
model=YOLO("best.pt")
result=model("pezzo_nuovo.jpg")
a questo punto il modello cercharà le tue classi, non soltanto quelle del modello originale.

Fino a pochi anni fa, implementare YOLO era un'impresa per pochi, bisognava scrivere script c++ e compilare file complessi. 
Ultralytics ha cambiato tutto, pensato come al sistema operativo della computer vision che permette di fare detection, segmentazioe e tracciamento senza mai cambiare il modo in cui scrivere il codice.

Architettura e Modelli
Non esiste una soluzione che va bene per tutto. Se devi far girare il modello su un piccolo drone utilizzi un modello diverso rispetto alla workstation potente in ufficio.

Una volta scelto il modello come faccio a renderlo operativo?

Al primo utilizzo, se i file dei pesi non sono presenti localmente, il framework li scarica dai server ufficiali, garantendo l'integrità dei modelli pre-addestrati su ImageNet o COCO
Il framework rileva automaticamente la presenza di core CUDA ottimizzando le prestazioni in tempo reale senza che noi dobbiamo muovere un dito. 
E' possibile forzare l'uso di un dispositivo specifico passando l'argomento 'device' durante l'inizializzazione (Device Management)

Oltre ai modelli locali, Ultralytics permette di caricare check point direttamente dal cloud, facilitando la collaborazione e il versionamento dei modelli.

Ma come fa il modello a capire se quello che vede è davvero l'oggetto o solo rumore?

La Logica del Confidence Score
Probabiltà e Localizzazione
Ogni predizone di YOLOv8 non restituisce solo una classe, ma un valore di confidenza che sintetizza la probabilità che l'oggetto esista in quel box e la precisione del box stesso.
Questo punteggio è fondamentale per filtrare il rumore di fondo, spacialmente in scene affollate dove il modello potrebbe generare falsi positivi a bassa probabilità

Il modello non ci dice mai: quello è un gatto sono sicuro, ci dice è una gatto, sono sicuro al 90% che qui ci sia un oggetto e che sia un gatto.

Predizione su Classi Standard
Il dataset COCO e l'inferenza
il dataset Common Object in Contet (COCO) è il benchmark mondiale per la detection, comprende 80 classi che spaziano dai pedoni agli oggetti domestici.
In questa fase vedremo come utilizzare il metodo 'predict' per processare immagini singolo o batch, sfruttando la conoscenza pre-acquisita dal modello durante il training ufficiale.

Il Metodo Predict
Eseguire la visione artificiale in una riga
- Source versatility: è possibile passare come sorgente un'immagine locale, un URL, un array numpy o persino una cartella intera.
- Argomento 'conf': parametro critico per impostare la soglia minima di confidenza (es. 0.25) sotto la quale le predizioni vengono scartate.
- Argomento 'iou': controlla la soglia per il Non-Maximum Suppression(NMS), dicendo quando due box sovrapposti appartengono allo stesso oggetto.
- Argument 'imgsz': permette di ridimensionare l'immagine in input prima dell'inferenza per bilanciare velocità e dettaglio.

Possiamo anche decidere cosa vogliamo vedere e cosa guardare

Il modello ha un dizionario interno dove le 80 classi sino associate a degli indici interi (da 0 a 79) che corrispondono a stringhe leggibli come 'macchina', 'persona', ecc.
Immagina di essere in una piazza affollata e di cercare solo le automobili, invece di analizzare ogni singola persona, possiamo dire a YOLO di concentrarsi solo su una specifica classe. 
Questo è possibile passando a YOLO una lista di ID all'argomento 'classes' durante la chiamata di .predict.
Questo filtro risparmia energia e pulisce l'output

Ma come fa il modello a scegliere tra due classi molto simili (es. un cane ed un lupo)?

Distribuzione della Softmax
Interpretazione del Class Probability

A differenza delle versioni precedenti, YOLOv8 utilizza un approccio anchor-free. La classificazione finale per ogni box è determinata da una distribuzione di probabilità sulla 80 classi COCO. La rete genera dei punteggi per ogni classe corrisondenti alla sua probabilità
Il punteggio finale per ogni classe viene normalizzato affinchè il modello possa esprimere incertezze tra categorie visivamente simili, come 'bicicletta' e 'motocicletta', la somma delle varie probabilità da 1.
La classe con il punteggio più alto 'vince' ma possiamo sempre curiosare tre le secondo/terze, ecc scelte, per capire se il modello era incerto.

Quindi il modello ha deciso, ma dove finiscono tutte queste informazioni?

Visualizzazione e Metadati
Estrarre Valore dai Risultati
L'inferenza produce un oggetto chiamnato 'Results' che contiene non solo l'immagine annotata, ma tutta la struttura dei dati grezzi necessari per analisi statistiche o integrazioni software.
Navigando in questi dati è possibile estrarre coordinate spaziali, classi e confidenza, i tempi di calcolo, in formato leggibile per Python (liste e tensori)
Leggere questi dati significa trasformare la visione in informazione.

Apriamo l'oggetto 'Results' 

Navighiamo l'Oggetto Results
Oltre i semplici rettangoli disegnati
All'interno troviamo l'attributo .boxes: contiene le coordinate dei bounding box in diversi formati come xyxy (vertici) o xywh (centro e dimensioni).
Se invece vuoi solo mostrare il risultato, il metodo plot() genera automaticamente un'immagine con box, etichette e confidenze già renderizzati, pronta per essere mostrata.
Se devi integrare YOLO su un APP medica o un sistema di visione, il metodo .numpy() e .tolist() estrae i dati dai tensori GPU per essere utilizzati in script standard e passati ad altri software

E' importante anche capire quanto tempo il sistema impiega per pensare

Loggin e Export Metadati
In produzione la velocità è tutto, l'oggetto 'Results' include anche i tempi di pre-processing, inferenza e post-processing, permettendo di calcolare gli FPS reali dell'applicazione.
Tempi suddivisi per ogni fase è utile perchè, se noti che il pre-processing è lento, potresti avere un collo di bottigli nel caricamento delle immagini.

Impostando save=True nel .predict viene creata una cartella runs/detect/predict dove archivia le immagini elaborate in modo sequenziale.

Natura delle coordinate salvate

Coordinate e Trasformazioni
Dal Pixel allo Spazio Relativo
Per gestire metadati in modo robusto, è spesso preferibile lavorare con coordinate normalizzate e non lavorare con i pixel. Questo evita che il cambio di risoluzione dell'immagine (esempio cambi la telecamera) rompa la logica di business dell'applicazione.
Per questo le best practies è utilzzare le coordinate normalizzate. Esprimere la posizione di un oggetto come una frazione tra 0 e 1 rende il software agnostico (indipendente) rispetto alla dimensione dell'immagine.
Se la foto è in 4k o a bassa risoluzione quell'oggetto si troverà sempre a quelle coordinate relative.
Il passaggio tra coordinate assolute (pixel) e relative è un'operazione lineare semplice ma cruciale per l'archiviazione in database.

In [ ]:
# python -m pip install -U ultralytics

In [1]:
import os

# --- CONFIGURAZIONE AMBIENTE 2026 ---
# Impostiamo Keras 3 per usare PyTorch come motore di calcolo.
# Questo garantisce interoperabilità nativa con i modelli YOLOv8 basati su Torch.
os.environ["KERAS_BACKEND"] = "torch"

import keras
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image
import requests
from io import BytesIO

class YOLOAdvancedPipeline:
    """
    Pipeline per la computer vision:
    1. Gestione dinamica di immagini da URL
    2. Inferenza scalabile con YOLOv8
    3. Visualizzazione avanzata dei metadati
    """

    def __init__(self, model_variant='yolov8n.pt'):
        """
        Inizializza il modello YOLO.
        Nel 2026, la variante 'nano' (.pt) è lo standard per l'edge computing.
        """
        print(f"Inizializzazione modello {model_variant}...")
        self.model = YOLO(model_variant)

    def download_image(self, url):
        """
        Scarica un'immagine da internet e la converte in formato gestibile.
        Utilizziamo un User-Agent per evitare blocchi dai server web.
        """
        print(f"Scaricamento immagine da: {url}")
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        # Convertiamo il contenuto scaricato in un'immagine PIL
        img = Image.open(BytesIO(response.content)).convert("RGB")
        return img

    def run_inference(self, image_source):
        """
        Esegue l'intero workflow: download -> predizione -> parsing.
        """
        # Caricamento immagine (se stringa inizia con http, la scarichiamo)
        if isinstance(image_source, str) and image_source.startswith("http"):
            image = self.download_image(image_source)
        else:
            image = image_source

        print("Esecuzione inferenza con parametri ottimizzati...")
        # 'conf=0.25': Filtro confidenza per eliminare il rumore
        # 'iou=0.7': Soglia NMS per la gestione delle sovrapposizioni
        # 'imgsz=640': Risoluzione standard per bilanciare velocità e mAP
        results = self.model.predict(
            source=image,
            conf=0.25,
            iou=0.7,
            imgsz=640,
            save=False
        )

        # Analizziamo il primo risultato del batch
        result = results[0]
        self._parse_metadata(result)
        self._visualize(result)

        return result

    def _parse_metadata(self, result):
        """Estrae i dati tensoriali e li converte in informazioni leggibili."""
        print(f"\n--- Metadati Rilevazione (Trovati {len(result.boxes)} oggetti) ---")
        
        for box in result.boxes:
            # Coordinate XYXY in pixel assoluti
            coords = box.xyxy[0].tolist()
            # Confidenza (sicurezza del modello)
            conf = float(box.conf[0])
            # Classe rilevata
            class_id = int(box.cls[0])
            class_name = self.model.names[class_id]

            print(f"[{class_name.upper()}] Conf: {conf:.2f} | Box: {[round(c, 1) for c in coords]}")

    def _visualize(self, result):
        """Renderizza i box sull'immagine e mostra il risultato finale."""
        # Il metodo .plot() restituisce l'immagine annotata in formato BGR (OpenCV)
        annotated_img = result.plot()

        # Conversione BGR -> RGB per una corretta visualizzazione con PIL/Matplotlib
        annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
        
        # Mostriamo il risultato
        final_view = Image.fromarray(annotated_img_rgb)
        
        # Nota: In ambiente locale questo apre il visualizzatore di sistema
        final_view.show()
        
        # Salvataggio su disco per persistenza
        final_view.save("ultimo_rilevamento.png")
        print("\nVisualizzazione generata e salvata come 'ultimo_rilevamento.png'")

# --- ESECUZIONE ---
if __name__ == "__main__":
    # Inizializziamo la pipeline
    pipeline = YOLOAdvancedPipeline()
    
    # URL di test (Esempio: autobus in ambiente urbano)
    URL_INTERNET = "https://ultralytics.com/images/bus.jpg"
    
    # Lanciamo il processo
    pipeline.run_inference(URL_INTERNET)

Creating new Ultralytics Settings v0.0.7 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\barbara\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Inizializzazione modello yolov8n.pt...
Scaricamento immagine da: https://ultralytics.com/images/bus.jpg
Esecuzione inferenza con parametri ottimizzati...

0: 640x480 4 persons, 1 bus, 1 stop sign, 18.4ms
Speed: 23.7ms preprocess, 18.4ms inference, 19.8ms postprocess per image at shape (1, 3, 640, 480)

--- Metadati Rilevazione (Trovati 6 oggetti) ---
[BUS] Conf: 0.87 | Box: [22.9, 231.3, 805.0, 756.8]
[PERSON] Conf: 0.87 | Box: [48.6, 398.6, 245.3, 902.7]
[PERSON] Conf: 0.85 | Box: [669.5, 392.2, 809.7, 877.0]
[PERSON] Conf: 0.83 | Box: [221.5, 405.8, 345.0, 857.5]
[PERSON] Conf: 0.26 | Box: [0.0, 550.5, 63.0, 873.4]
[STOP SIGN] Conf: 0.26 | Box: [0.1, 254.5, 32.